## Pydaytic
- pydantic.BaseModel을 상속받는 클래스로 모델 생성

In [ ]:
from pydantic import BaseModel

# 문서 한 개를 담을 수 있는 클래스 생성
class Document(BaseModel):
    doc_id:str
    title:str
    version:str
    security_level:str

doc=Document(
    doc_id='DOC-HR-001',
    title='Confidential Document',
    version='1.0',
    security_level='authorized access only'
)

print(doc) # __repr__도 자동으로 정의해준다.
print(doc.title)

doc_id='DOC-HR-001' title='Confidential Document' version='1.0' security_level='authorized access only'
Confidential Document


In [14]:
from pydantic import Field
# 제약이 붙은 문서 모델
class Document2(BaseModel):
    doc_id:str=Field(
        ...,
        pattern=r'DOC-[A-Z]{2,4}-\d{3}$',
        description='Document Serial Number'
    )
    title:str=Field(
        ...,
        min_length=1,
        max_length=200,
    )
    version:str=Field(..., pattern=r'^\d+\.\d+$') # 2.0
    security_level:str
    tags:list[str]=Field(default_factory=list)

doc2=Document2(
    doc_id='DOC-HR-002',
    title='Buisness Trip Budget Regulation',
    version='1.2',
    security_level='Internal',
    tags=['aaa','bbb']
)
print(doc2)
print(doc2.tags)

doc_id='DOC-HR-002' title='Buisness Trip Budget Regulation' version='1.2' security_level='Internal' tags=['aaa', 'bbb']
['aaa', 'bbb']


In [15]:
from enum import Enum
from typing import Literal

class SecurityLevel(str,Enum):
    PUBLIC='전체 공개'
    INTERNAL='사내 공개'
    CONFIDENTIAL='대외비'
    SECRET='기밀'

class Document3(BaseModel):
    doc_id:str
    security_level:SecurityLevel # Enum 타입으로 지정
    sec_level:Literal['전체 공개','사내 공개','대외비','기밀']

doc3=Document3(doc_id='DOC-3', security_level=SecurityLevel.CONFIDENTIAL, sec_level='대외비')

print(doc3)
print(doc3.security_level.value)

doc_id='DOC-3' security_level=<SecurityLevel.CONFIDENTIAL: '대외비'> sec_level='대외비'
대외비


In [21]:
from datetime import date

class Document4(BaseModel):
    doc_id:str
    title:str
    effective_date:date
    tags:list[str]=Field(default_factory=list)

doc4=Document4(
    doc_id='DOC-HR-004',
    title='Inland Buisness Trip Budget Regulation',
    effective_date=date(2026,9,7),
    tags=['BusinessTrip','Budget','Regulation']
)

print(doc4)

# 파이썬 객체인 모델을 딕셔너리로 변경
d=doc.model_dump()
print(d)

# 파이썬 객체인 모델을 JSON으로 변경
j=doc.model_dump_json(indent=2) # indent: 들여쓰기 옵션
print(j)

doc_id='DOC-HR-004' title='Inland Buisness Trip Budget Regulation' effective_date=datetime.date(2026, 9, 7) tags=['BusinessTrip', 'Budget', 'Regulation']
{'doc_id': 'DOC-HR-001', 'title': 'Confidential Document', 'version': '1.0', 'security_level': 'authorized access only'}
{
  "doc_id": "DOC-HR-001",
  "title": "Confidential Document",
  "version": "1.0",
  "security_level": "authorized access only"
}


In [25]:
# dict/json -> 파이썬 객체 모델

# dict type
data={
    'doc_id':'DOC-SEC-002',
    'title':'Info Security Rule',
    'effective_date':'2026-09-07',
    'tags':['security']
}
doc_data=Document4.model_validate(data)
print(data)
print(type(doc_data))

# JSON 데이터
data_json = '{"doc_id": "DOC-SEC-002","title" : "정보 보안 지침","effective_date": "2026-09-07", "tags": ["보안"]}'
doc_json = Document4.model_validate_json(data_json)
print(doc_json)

{'doc_id': 'DOC-SEC-002', 'title': 'Info Security Rule', 'effective_date': '2026-09-07', 'tags': ['security']}
<class '__main__.Document4'>
doc_id='DOC-SEC-002' title='정보 보안 지침' effective_date=datetime.date(2026, 9, 7) tags=['보안']


In [ ]:
from pydantic import BaseModel, Field, field_validator
from datetime import date, datetime
from enum import enum

# 보안 등급: 여러 파일에서 사용하므로 enum으로 고정
class SecurityLevel(str, Enum):
    PUBLIC='공개'
    INTERNAL='사내공개'
    CONFIDENTIAL='대외비'
    SECRET='기밀'

# 요청 모델: 문서 등록 요청-> 클라이언트가 보내는 것
class DocumentCreate(BaseModel):
    doc_id:str=Field(..., pattern=r'DOC-[A-Z]{2,4}-\d{3}$', description='문서 고유 번호 (예: DOC-HR-001)')
    title:str=Field(..., min_length=1, max_length=200)
    version:str=Field(..., pattern=r'^\d+\.\d+$', description='예: 1.0')
    department:str=Field(..., min_length=1, max_length=50)
    security_level:securityLevel
    effective_date:date
    expiry_date:date | None=None

    # doc_id가 소문자로 들어와도 대문자로 저장하는 처리
    @field_validator('doc_id', mode='before')
    @classmethod
    def upper_doc_id(cls,v):
        return v.upper() is isinstance(v,str) else v

    # title과 department의 앞뒤 공백 제거 처리 (여러 필드 지정 가능)
    @field_validator('title','department')
    @classmethod
    def strip_text(cls, v:str)->str:
        return v.strip()

# 응답 모델: 문서 응답 - 서버가 돌려주는 데이터 형태
class DocumentOut(BaseModel):
    id:int                       # 서버에서 사용될 데이터 고유 번호
    doc_id: str
    title: str
    version: str
    department: str
    effective_date: date
    expiry_date: date | None
    is_latest: bool              # 최신본 여부
    chunk_count: int=Field(ge=0) # 임베딩 된 조각 수
    created_at: datetime         # 문서 저장 날짜